In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
def build_daily_compensation_fact():
    """
    Constructs the Daily Business Compensation Fact table for the 2023 fiscal year.
    
    This function implements a 'Point-in-Time' join to ensure businesses are 
    compensated based on their location and size at the time of a street closure.
    
    Processing Steps:
    1. Temporal Explosion: Expands closure start/end ranges into individual daily rows.
    2. Master Lookup: Resolves street names to 'street_id' via dim_street for Star Schema integrity.
    3. Historical Join: Links closures to businesses using SCD2 logic (effective dates) 
       with a fallback to 'is_current' records to handle backfilling for 2023.
    4. Measures Calculation: 
       - Multiplies area by a rate of 100 ILS.
       - Caps total daily payout at 10,000 ILS per Rule #4.
    5. Deduplication: Ensures a single payout per business, per day, even if 
       multiple closure records overlap.

    Returns:
        str: Success message with a count of the generated fact records.
    """
    
    # Configuration
    dim_biz_path = "workspace.tel_aviv_municipality.dim_business"
    dim_st_path = "workspace.tel_aviv_municipality.dim_street"
    stg_closures_path = "workspace.tel_aviv_municipality_stg.street_closures"
    target_table = "workspace.tel_aviv_municipality.fact_2023_daily_business_compensation"

    dim_biz = spark.table(dim_biz_path)
    dim_st = spark.table(dim_st_path)
    stg_closures = spark.table(stg_closures_path)

    # Prepare Closure Events (Daily Grain for 2023)
    closures_daily = stg_closures \
        .withColumn("start_dt", F.to_date(F.col("start_at"))) \
        .withColumn("end_dt", F.to_date(F.col("end_at"))) \
        .filter((F.year("start_dt") == 2023) | (F.year("end_dt") == 2023)) \
        .withColumn("date", F.explode(F.sequence("start_dt", "end_dt"))) \
        .filter(F.year("date") == 2023) \
        .select(F.col("closure_id"), F.col("street_name"), F.col("date"))

    # Map Name to ID (Star Schema Bridge)
    closures_with_id = closures_daily.alias("c") \
        .join(dim_st.alias("s"), "street_name", "inner") \
        .select("c.closure_id", "c.date", "c.street_name", "s.street_id")

    # Historical Join to Businesses (ID -> ID)
    # logic handles both historical records and backfills using 'is_current'
    fact_joined = closures_with_id.alias("c") \
        .join(dim_biz.alias("b"), "street_id") \
        .where("""
            (c.date >= CAST(b.effective_from AS DATE) AND (c.date < CAST(b.effective_to AS DATE) OR b.effective_to IS NULL))
            OR (b.is_current = true)
        """)

    # Apply Payout Rules and Deduplicate
    # Bussiness logic: ensures 1 payout per day per business 
    window_spec = Window.partitionBy("b.business_id", "c.date") \
        .orderBy(F.col("b.is_current").desc(), F.col("b.effective_from").desc())

    fact_final = fact_joined.withColumn("row_num", F.row_number().over(window_spec)) \
        .filter(F.col("row_num") == 1) \
        .withColumn("rate", F.lit(100)) \
        .withColumn("daily_compensation", 
            F.least(F.lit(10000.0), F.coalesce(F.col("b.house_area_sqm"), F.lit(0)) * 100)
        ) \
        .select(
            F.col("c.date"),
            F.col("b.business_id"),
            F.col("b.primary_holder_name").alias("business_name"),
            F.col("b.type").alias("business_type"),
            F.col("c.street_name"),
            F.col("b.house_area_sqm").alias("area_sqm"),
            F.col("rate"),
            F.col("daily_compensation").alias("daily_compensation_ils"),
            F.current_timestamp().alias("fact_ingestion_at")
        )

    try:
        fact_final.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)
        
        count = spark.table(target_table).count()
        return f"Success: Built {target_table}. Total rows: {count}"
    
    except Exception as e:
        return f"Error building Fact Table: {str(e)}"



In [0]:
# Execute
print(build_daily_compensation_fact())